In [ ]:
%matplotlib inline

from os import path
import os
import sys
# Add the parent directory to sys.path
sys.path.insert(0, path.abspath('../'))

from gpbayestools.bayesian_analysis import BayesianAnalysis

import logging
# show the progress messages of gpbayestools
logging.basicConfig(level=logging.INFO, format="[%(levelname)s][%(module)s] %(message)s")


In [ ]:
exp_path = "./separate_training_posterior_data/example_data_test.pkl"
model_par = "../data/modelDesign_3DMCGlauber.txt"
mymcmc = BayesianAnalysis(exp_data_path=exp_path, parameter_file=model_par)


In [ ]:
folder = "./emulators_full_trained/trained_emulators/"
emuPathList = [folder+"AuAu200_dNdy_train_PCSK_trained.sav",
               folder+"AuAu200_PHOBOSdNdeta_train_PCSK_trained.sav",
               folder+"AuAu200_PHOBOSv2eta_train_PCSK_trained.sav",
               folder+"AuAu200_pTvn_train_PCSK_trained.sav",
               folder+"AuAu19p6_dNdy_train_PCSK_trained.sav",
               folder+"AuAu19p6_PHOBOSdNdeta_train_PCSK_trained.sav",
               folder+"AuAu19p6_pTvn_train_PCSK_trained.sav",
               folder+"AuAu7.7_dNdy_train_PCSK_trained.sav",
               folder+"AuAu7.7_pTvn_train_PCSK_trained.sav",
               ]
mymcmc.load_emulators(emuPathList)

In [ ]:
os.environ["OMP_NUM_THREADS"] = "1"
# may have to: export RDMAV_FORK_SAFE=1 before running the code
os.environ["RDMAV_FORK_SAFE"] = "1"

n_effective=8000
n_active=4000
n_prior=16000
sample="tpcn"
n_max_steps=100
random_state=42

n_total = 30000
n_evidence = 5000

pool = 12

mymcmc.run_pocomc(n_effective=n_effective, n_active=n_active,
                  n_prior=n_prior, sample=sample,
                  n_max_steps=n_max_steps, random_state=random_state,
                  n_total=n_total, n_evidence=n_evidence, pool=pool)


In [ ]:
n_steps = 1000
n_walkers = 100
n_burn_steps = 7000
status = None
n_thin = 2
mymcmc.run_emcee(n_steps=n_steps, n_burn_steps=n_burn_steps,
                 n_walkers=n_walkers, status=status, n_thin=n_thin,
                 skip_initial_state_check=True)

In [ ]:
n_steps = 1000
n_walkers = 100
n_temps = 30
max_temp = 100
n_start_parameters = 7000

mymcmc.run_ptlmc(n_steps=n_steps, n_walkers=n_walkers, n_temps=n_temps,
                 max_temp=max_temp, n_start_parameters=n_start_parameters)

In [ ]:
# compute the log likelihood for each point of the chain of one sampler and
# print the shapes; each sampler writes its own chain file
import pickle
sampler_name = "pocoMC"  # 'emcee', 'pocomc' or 'ptlmc'
mymcmc.compute_log_likelihood_for_chain(sampler=sampler_name)
chain_file = mymcmc.chain_path(sampler_name)
with open(chain_file, "rb") as f:
    chain = pickle.load(f)
with open(chain_file.with_name(chain_file.stem + "_log_likelihood.pkl"), "rb") as f:
    log_likelihood = pickle.load(f)
print(chain['chain'].shape)
print(log_likelihood['log_likelihood'].shape)